# Codegen
CodeGen is a family of autoregressive language models for program synthesis from the paper: A Conversational Paradigm for Program Synthesis by Erik Nijkamp, Bo Pang, Hiroaki Hayashi, Lifu Tu, Huan Wang, Yingbo Zhou, Silvio Savarese, Caiming Xiong. 

In this Notebook we are going to load the codegen-2B-mono model where:
* **2B** stands for the amount of parameters of the model
* **mono** stands on what is trained on, in this case mono means that the model was trainded for python code generation only.

In [ ]:
# PIP INSTALLS
# These will eventually be already installed in the container, so there should be no pip install inside the notebook
!pip install accelerate
#Transformers needs to be updated
!pip install  transformers
#Install our qaic apis
!python3 -m pip install /opt/qti-aic/dev/lib/x86_64/qaic-0.0.1-py3-none-any.whl
!pip install torch
!pip install numpy
!pip install onnx

In [ ]:
!pip install tensorflow

## Imports

In [ ]:
import qaic
import numpy as np
import tensorflow as tf
import os
import shutil
from pathlib import Path
import torch
from torch.onnx import export
import onnx
from packaging import version

We are going to use the Transformer library from huggingface.

In [3]:
from transformers import CodeGenForCausalLM, AutoTokenizer

/local/mnt/workspace/fbuoncom/venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
tokenizer = AutoTokenizer.from_pretrained('./cd/tokenizer')

## Download the model

In [ ]:
# # Load the model and tokenizer
checkpoint = "Salesforce/codegen-2B-mono"
model = CodeGenForCausalLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model.save_pretrained("./cd/model",max_shard_size='20GB')
tokenizer.save_pretrained("./cd/tokenizer")
del model
del tokenizer

## First generation of the ONNX


In [ ]:
@torch.no_grad()
def convert_models(model_path: str, output_path: str, opset: int):
    print("Loading the model")
    model = CodeGenForCausalLM.from_pretrained(model_path)
    output_path = Path(output_path)
    output_model_path = output_path / "model.onnx"
    print("Loaded the model")
    
    # Dicts
    input_names = ["input_ids"]
    output_names = ["output"]
    dynamic_axes = {
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "output": {0: "batch_size", 1: "sequence_length"},
    }
    
    print("Creating the dummy input")
    text = "def dummy_input:"
    # Make the pad token equal to the eos token
    tokenizer.pad_token = tokenizer.eos_token
    tokens = tokenizer(text,
                       padding='max_length',
                       max_length=128,
                       return_tensors="pt")
    
    print("Created the dummy input")
    print("Starting the export")
    torch.onnx.export(
        model,
        tokens['input_ids'],
        f=output_model_path.as_posix(),
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        do_constant_folding=True,
        opset_version=opset
    )
    print("Finished the export")
    
    output_model_path = str(output_model_path.absolute().as_posix())
    output_model_dir = os.path.dirname(output_model_path)
    model = onnx.load(output_model_path)
    # clean up existing tensor files
    shutil.rmtree(output_model_dir)
    os.mkdir(output_model_dir)
    
    print("Started the squashing of the model")
    onnx.save_model(
        model,
        output_model_path,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location="weights.pb",
        convert_attribute=False,
    )
    print("Squashed model")
    del model

In [ ]:
# Convert the UNet with ONNX opset 14
convert_models('./cd/model','./cd/model_onnx',14)
print("Finished")

In [4]:
onnx_model_path="./cd/model_onnx/model.onnx"
qpc_model_path="./cd/model_qpc/programqpc.bin"

We will use the qaic.Session to create our qpc.

Let's analyze the various parameters passed to the qaic.Session() constructor.

| Parameter | Explanation |
| --- | --- |
| **dev_id** | The device id we are going to use for our calculation|
| **aic_num_cores** | 	Number of aic cores to be used for inference |
| **convert_to_fp16** | Run all floating-point in fp16 |

In [6]:
qaic_sess = qaic.Session(onnx_model_path,
                         dev_id=0,
                         aic_num_cores=14,
                         convert_to_fp16=True,
                         onnx_define_symbol={"batch_size":1,
                                             "sequence_length":128},
                         output_dir="./cd/model_qpc/"
                        )

Options yaml file available at:  ./cd/model_qpc//options.yaml


Now we have our Session ready we can run inference, let's create an example prompt for a hello word function.
We then must use the tokenizer, as the input of our model will be an already tokenized text.

## Prepare our input text

In [5]:
text = "# Class definition of a Dog"
tokens_no_pad = tokenizer(text,
                   return_tensors="pt")['input_ids']
initial_token_size=int(tokens_no_pad.shape[1])

In [6]:
tokenizer.pad_token = tokenizer.eos_token
tokens = tokenizer(text,
                   padding='max_length',
                   max_length=128,
                   return_tensors="pt")

In [7]:
tokens['input_ids']=tokens['input_ids'].cpu().numpy()
starting_inputs = tokens['input_ids']

## Load the QPC on the card

In [8]:
qpc_model_path = "./cd/model_qpc/programqpc.bin"
yaml_options_path = "./cd/model_qpc/options.yaml"
qaic_sess = qaic.Session(qpc_model_path, options_path=yaml_options_path)

## Inference


In [9]:
def generate_qaic_output(qaic_sess, original_input_ids, initial_token_size, max_new_tokens):
    cur_len = initial_token_size
    input_ids = original_input_ids

    while cur_len < max_new_tokens:
        logits = qaic_sess.run({"input_ids":input_ids})['output']
        logits = logits.reshape(1,128,51200)
        
        # Get the logits of the last token
        next_token_logits = logits[:, cur_len-1, :]

        # Choose the token with the highest probability
        next_token = np.argmax(next_token_logits, axis=1)[:, np.newaxis]

        # Append next_token at the end of input_ids
        input_ids[0][cur_len]=next_token
        cur_len += 1
            
    return input_ids

In [10]:
output_tokens = generate_qaic_output(qaic_sess,starting_inputs,initial_token_size,80)

qaic: WARNING: Acitvating network, this will add up to time of first inference


In [11]:
decoded_string = tokenizer.decode(output_tokens[0])

In [12]:
print(decoded_string)

# Class definition of a Dog
#
# Author: David Fisher
# Course: CS151
# Date Created: 2020-02-20
# Date Revised: 2020-02-20

class Dog:
    """
    Class Dog that represents a dog.
    """

    def __init__(self, name, age, breed, spots):
        """<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
